In [1]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "sentence-transformers", "torchaudio", "transformers", "trl"],
               capture_output=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/huggingface/transformers.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/huggingface/trl.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "peft>=0.19.0", "accelerate>=1.2.1",
                "bitsandbytes>=0.45.0", "torchvision>=0.19.0"], check=True)

print("✅ Cài đặt hoàn tất! Vào Menu > Run > Restart Session, rồi chạy từ Cell 2.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 34.2 MB/s eta 0:00:00
✅ Cài đặt hoàn tất! Vào Menu > Run > Restart Session, rồi chạy từ Cell 2.


In [2]:
import os
import torch
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoModelForImageTextToText, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    VaccineNLP_TOKEN = user_secrets.get_secret("VaccineNLP")
    login(token=VaccineNLP_TOKEN)
    print("✅ Đăng nhập HuggingFace thành công!")
except Exception as e:
    print(f"⚠️ Không tìm thấy VaccineNLP: {e}")


✅ Đăng nhập HuggingFace thành công!


In [3]:
TRAIN_PATH      = "/kaggle/input/datasets/inhlqunhphng/vaccinenlp-clean-data/04_silver_labels/train_set_final.jsonl"
TEST_PATH       = "/kaggle/input/datasets/inhlqunhphng/vaccinenlp-clean-data/03_processed/benchmark_test_set.jsonl"
MODELS_SAVE_DIR = "/kaggle/working/gemma_qlora_xai"
os.makedirs(MODELS_SAVE_DIR, exist_ok=True)

for path in [TRAIN_PATH, TEST_PATH]:
    print(("✅" if os.path.exists(path) else "❌ KHÔNG TÌM THẤY") + f": {path}")
print(f"\n📁 Lưu model: {MODELS_SAVE_DIR}")


✅: /kaggle/input/datasets/inhlqunhphng/vaccinenlp-clean-data/04_silver_labels/train_set_final.jsonl
✅: /kaggle/input/datasets/inhlqunhphng/vaccinenlp-clean-data/03_processed/benchmark_test_set.jsonl

📁 Lưu model: /kaggle/working/gemma_qlora_xai


In [4]:
model_id = "google/gemma-4-E4B-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

misinfo_map   = {0: "Không liên quan", 1: "Tin giả", 2: "Chính xác"}
stance_map    = {0: "Ủng hộ", 1: "Phản đối", 2: "Trung lập", 3: "Không rõ"}
sentiment_map = {0: "Tiêu cực", 1: "Trung tính", 2: "Tích cực"}

def format_xai_prompt(row):
    text = row.get('text_cleaned') or row.get('text') or row.get('text_original') or str(list(row.values())[0])
    reasoning = row.get('llm_reasoning', 'Phân tích dựa trên ngữ cảnh.')
    ids = row.get('standardized_ids')
    if not ids or len(ids) != 3: ids = [0, 3, 1]

    user_prompt = (
        f"You are an Explainable AI in Public Health. Analyze the text, "
        f"provide your reasoning first, and then the structured labels.\n\nVăn bản: {text}"
    )
    assistant_response = (
        f"Lý luận: {reasoning}\n"
        f"Kết quả: {misinfo_map.get(ids[0], 'Không liên quan')} | "
        f"{stance_map.get(ids[1], 'Không rõ')} | "
        f"{sentiment_map.get(ids[2], 'Trung tính')}"
    )
    messages = [
        {"role": "user",  "content": user_prompt},
        {"role": "model", "content": assistant_response}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

print("Đang tải và map dữ liệu...")
train_ds = load_dataset("json", data_files=TRAIN_PATH, split="train").map(format_xai_prompt)
test_ds  = load_dataset("json", data_files=TEST_PATH,  split="train").map(format_xai_prompt)
print("✅ Format Data hoàn tất!")
print(train_ds[0]['text'][:300] + "...")


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Đang tải và map dữ liệu...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1670 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/186 [00:00<?, ? examples/s]

✅ Format Data hoàn tất!
<bos><|turn>user
You are an Explainable AI in Public Health. Analyze the text, provide your reasoning first, and then the structured labels.

Văn bản: deputy prime minister vũ đức đam volunteers to take the vietnamese covid vaccine. still experimental.<turn|>
<|turn>model
Lý luận: Phân tích dựa trên...


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# In top-level modules để debug khi cần
top_modules = sorted(set(n.split('.')[0] for n, _ in model.named_modules() if n))
print("Top-level modules:", top_modules)

# FIX: PEFT >= 0.19.0 với "all-linear" + exclude_modules tự bỏ qua ClippableLinear
# Regex "language_model.*" bị lỗi vì tên module thực tế có thể khác nhau
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    exclude_modules=["vision_tower", "audio_tower", "multi_modal_projector"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()
print("✅ Khởi tạo QLoRA hoàn tất.")


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Top-level modules: ['lm_head', 'model']
trainable params: 50,499,584 || all params: 7,991,600,416 || trainable%: 0.6319
✅ Khởi tạo QLoRA hoàn tất.


In [6]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir=MODELS_SAVE_DIR,
    max_length=256,
    dataset_text_field="text",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=2,
    save_strategy="epoch",
    eval_strategy="no",                # FIX OOM: tắt eval sau mỗi epoch
    optim="paged_adamw_8bit",
    bf16=True,
    max_grad_norm=0.3,
    warmup_steps=10,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    ddp_find_unused_parameters=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    args=sft_config,
)

print("🚀 Bắt đầu huấn luyện...")
torch.cuda.empty_cache()
trainer.train()

trainer.model.save_pretrained(f"{MODELS_SAVE_DIR}/final_model")
tokenizer.save_pretrained(f"{MODELS_SAVE_DIR}/final_model")
print(f"💾 Đã lưu model tại: {MODELS_SAVE_DIR}/final_model")


Adding EOS to train dataset:   0%|          | 0/1670 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1670 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/186 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/186 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


🚀 Bắt đầu huấn luyện...


Step,Training Loss
10,4.543576
20,1.743194
30,1.652146
40,1.558271
50,1.412548
60,1.365537
70,1.483570
80,1.547689
90,1.480017
100,1.535907


💾 Đã lưu model tại: /kaggle/working/gemma_qlora_xai/final_model
